# Team MLB · 실험 6. Soft voting 앙상블

LR·SVM·RF의 확률을 같은 가중치로 평균해 단독 모델보다 나아지는지 본다.

| 순서 | 내용 |
| --- | --- |
| 0 | 설정 · 경로 |
| 1 | 입력 확률 파일 형식 |
| 1-1 | RF 확률 파일(npz) → CSV 변환 |
| 2 | 확률 파일 확인 |
| 3 | 공통 함수 |
| 4 | 앙상블 실행 · 저장 |

- 확률 파일이 있는 모델이 2개 이상인 과제만 합친다.
- 결합 전에 모델끼리 **record_id·정답·클래스 열이 같은지** 검사하고, 다르면 멈춘다.
- fold 배정은 모델마다 달라도 된다(경고만 출력). 각 OOF 확률은 그 모델이 해당 기록을 학습하지 않은 상태의 값이라 평균 결합에는 문제가 없다.
- 가중치는 모두 같게 두고 튜닝하지 않는다.
- 이진 분류는 실험 5와 같은 기준(검증 데이터로 정한 목표 오탐률의 임계값)으로 비교한다.

먼저 실행할 노트북: `models/logistic_regression/01_lr_main.ipynb`(LR 확률), `ensemble/MLB_SVM_probability.ipynb`(SVM 확률), `models/random_forest/MLB_RF_Experiment3_4_5.ipynb`(RF 확률 npz).

## 0. 설정

In [1]:
import os, sys, subprocess, itertools
from pathlib import Path

def find_repo_root():
    """저장소 루트(splits/ 폴더가 있는 위치)를 찾는다. Colab에서 노트북만 연 경우 저장소를 내려받는다."""
    here = Path.cwd()
    for p in [here, *here.parents]:
        if (p / "splits" / "split_seed42.csv").exists():
            return p
    if "google.colab" in sys.modules:
        dst = Path("/content/website-fingerprinting")
        if not dst.exists():
            subprocess.run(["git", "clone", "-q", "https://github.com/EWHA-MLB/website-fingerprinting.git", str(dst)], check=True)
        return dst
    raise FileNotFoundError("저장소 안에서 실행해야 한다 (splits/ 폴더를 찾지 못함)")

ROOT = find_repo_root()
RESULTS_ROOT = ROOT / "results"
MODEL_DIRS = {
    "lr":  str(RESULTS_ROOT / "logistic_regression"),
    "svm": str(RESULTS_ROOT / "svm"),
    "rf":  str(RESULTS_ROOT / "ensemble" / "rf_converted"),     # 1-1에서 변환한 RF 확률
}
ENSEMBLE_DIR = str(RESULTS_ROOT / "ensemble")

# RF 노트북(MLB_RF_Experiment3_4_5)이 확률 npz를 저장한 실행 폴더
RF_RUN_DIR = str(RESULTS_ROOT / "rf_experiments3_4_5_110features" / "runs" / "29c10470afbc6158f597ca2749dac781")
RF_TASK_DIR = {"closed": "closed_world", "multi": "open_multi", "binary": "open_binary"}
TARGET_FPR  = [0.01, 0.05, 0.10, 0.20]     # 실험 5와 같은 목표 오탐률
TASKS       = ("closed", "multi", "binary")


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, roc_auc_score
pd.set_option("display.max_columns", 50)
os.makedirs(ENSEMBLE_DIR, exist_ok=True)
os.makedirs(MODEL_DIRS["rf"], exist_ok=True)

## 1. 입력 확률 파일 형식

세 모델 모두 같은 형식으로 저장한다. LR은 `01_lr_main.ipynb`의 실험 5 준비 셀, SVM은 `MLB_SVM_probability.ipynb`가 만든다.

| 파일 | 열 |
| --- | --- |
| `results/{모델 폴더}/{모델}_oof_{과제}.csv` | `record_id, y, fold, p_클래스…` |
| `results/{모델 폴더}/{모델}_test_proba_{과제}.csv` | `record_id, y, p_클래스…` |

- 분할은 `splits/split_seed42.csv`, fold는 `splits/cv_folds_seed42.csv`를 쓴다.
- 행은 `record_id` 순으로 정렬한다.
- 하이퍼파라미터는 각 모델의 실험 4에서 고른 값을 쓰고, 평가 데이터로 다시 고르지 않는다.
- 과제별 정답: `closed` 0~94 / `binary` 1(mon)·-1(unmon) / `multi` 0~94, unmon -1

## 1-1. RF 확률 파일(npz) → CSV 변환

RF 노트북은 `oof_probabilities.npz`(record_id, y_true, cv_fold, proba, classes)와 `test_probabilities.npz`(record_id, proba, classes)로 저장한다. 같은 CSV 형식으로 바꿔 `results/ensemble/rf_converted/`에 저장하고, RF 원본 폴더는 건드리지 않는다. test npz에는 정답이 없어서 LR test 파일의 정답을 record_id로 붙인다.

In [3]:
for task, sub in RF_TASK_DIR.items():
    src = os.path.join(RF_RUN_DIR, sub)
    f_oof, f_te = os.path.join(src, "oof_probabilities.npz"), os.path.join(src, "test_probabilities.npz")
    if not (os.path.exists(f_oof) and os.path.exists(f_te)):
        print(f"{task}: RF npz 없음 → 건너뜀 ({src})"); continue
    o, t = np.load(f_oof), np.load(f_te)
    cols = [f"p_{c}" for c in o["classes"]]
    assert list(o["classes"]) == list(t["classes"]), f"{task}: OOF/test 클래스 순서 다름"

    oof = pd.DataFrame(o["proba"], columns=cols)
    oof.insert(0, "record_id", o["record_id"]); oof.insert(1, "y", o["y_true"]); oof.insert(2, "fold", o["cv_fold"])

    lr_te = pd.read_csv(os.path.join(MODEL_DIRS["lr"], f"lr_test_proba_{task}.csv"), usecols=["record_id", "y"])
    te = pd.DataFrame(t["proba"], columns=cols); te.insert(0, "record_id", t["record_id"])
    te = te.merge(lr_te, on="record_id", how="left")
    assert te.y.notna().all(), f"{task}: LR test에 없는 record_id가 있음"
    te = te[["record_id", "y"] + cols]

    oof.to_csv(os.path.join(MODEL_DIRS["rf"], f"rf_oof_{task}.csv"), index=False)
    te.to_csv(os.path.join(MODEL_DIRS["rf"], f"rf_test_proba_{task}.csv"), index=False)
    print(f"{task}: OOF {len(oof):,}행 · test {len(te):,}행 변환 완료")

closed: OOF 15,200행 · test 3,800행 변환 완료
multi: OOF 23,200행 · test 5,800행 변환 완료
binary: OOF 23,200행 · test 5,800행 변환 완료


## 2. 확률 파일 확인

모델·과제별로 확률 파일이 있는지 표시한다(O = 있음). O가 2개 이상인 과제만 4번에서 합친다.

In [4]:
def prob_paths(m, task):
    folder = MODEL_DIRS[m]
    return (os.path.join(folder, f"{m}_oof_{task}.csv"),
            os.path.join(folder, f"{m}_test_proba_{task}.csv"))

status = pd.DataFrame(
    {task: {m: "O" if all(os.path.exists(p) for p in prob_paths(m, task)) else "-" for m in MODEL_DIRS}
     for task in TASKS})
display(status)

,closed,multi,binary
lr,O,O,O
svm,O,O,O
rf,O,O,O


## 3. 공통 함수

In [5]:
def threshold_at_fpr(y, p, target):
    # 검증 데이터의 unmon 점수에서 오탐률이 target 이하가 되는 임계값 (p > t 이면 mon)
    return float(np.quantile(p[y == -1], 1 - target, method="higher"))

def rates(y, p, t):
    pred = p > t
    tp = int((pred & (y == 1)).sum());  fp = int((pred & (y == -1)).sum())
    fn = int((~pred & (y == 1)).sum()); tn = int((~pred & (y == -1)).sum())
    return dict(TPR=tp / (tp + fn), FPR=fp / (fp + tn),
                precision=tp / (tp + fp) if tp + fp else np.nan, acc=(tp + tn) / len(y))

def threshold_table(y_v, p_v, y_t, p_t):
    cands = [("기본 (p>0.5)", 0.5)] + [(f"검증 FPR≤{f:.2f}", threshold_at_fpr(y_v, p_v, f)) for f in TARGET_FPR]
    rows = []
    for name, t in cands:
        v, s = rates(y_v, p_v, t), rates(y_t, p_t, t)
        rows.append({"기준": name, "임계값": t,
                     "검증 TPR": v["TPR"], "검증 FPR": v["FPR"],
                     "test TPR": s["TPR"], "test FPR": s["FPR"], "test Precision": s["precision"],
                     "test 정확도": s["acc"]})
    return pd.DataFrame(rows)

def load_model_probs(task):
    got = {}
    for m in MODEL_DIRS:
        p_oof, p_te = prob_paths(m, task)
        if os.path.exists(p_oof) and os.path.exists(p_te):
            got[m] = (pd.read_csv(p_oof).sort_values("record_id").reset_index(drop=True),
                      pd.read_csv(p_te).sort_values("record_id").reset_index(drop=True))
    return got

def check_aligned(got):
    ref_oof, ref_te = next(iter(got.values()))
    cols = sorted([c for c in ref_oof.columns if c.startswith("p_")], key=lambda c: int(c[2:]))
    for m, (o, t) in got.items():
        for name, d, r in (("OOF", o, ref_oof), ("test", t, ref_te)):
            assert len(d) == len(r) and (d.record_id.values == r.record_id.values).all(), f"{m} {name}: record_id 불일치"
            assert (d.y.values == r.y.values).all(), f"{m} {name}: 정답(y) 불일치"
            assert set(c for c in d.columns if c.startswith("p_")) == set(cols), f"{m} {name}: 클래스 열 불일치"
        if not (o.fold.values == ref_oof.fold.values).all():
            print(f"  [참고] {m}: fold 배정이 기준 모델과 다름 (평균 결합에는 문제없음, 스태킹 등에는 사용 불가)")
    return cols

def evaluate(task, y, P, classes):
    pred = classes[P.argmax(1)]
    out = {"정확도": (pred == y).mean(), "Macro-F1": f1_score(y, pred, average="macro")}
    if task == "multi":
        is_mon = y >= 0
        out["mon 사이트 정답"] = (pred[is_mon] == y[is_mon]).mean()
        out["mon→mon"] = (pred[is_mon] >= 0).mean()
        out["unmon→unmon"] = (pred[~is_mon] == -1).mean()
    if task == "binary":
        out["ROC-AUC"] = roc_auc_score(y, P[:, list(classes).index(1)])
    return out

## 4. 앙상블 실행 · 저장

- 단독 모델 성능도 같은 확률 파일에서 다시 계산해 앙상블과 같은 조건으로 비교한다.
- 2개 조합과 3개 조합을 모두 만들고 확률을 같은 비율로 평균한다.
- 결과는 `results/ensemble/`에 저장한다.

In [6]:
ENS = {}
for task in TASKS:
    got = load_model_probs(task)
    print(f"\n===== {task} | 확률 파일이 있는 모델: {list(got) or '없음'}")
    if len(got) < 2:
        print("  확률 파일이 있는 모델이 2개 미만이라 건너뜀")
        continue
    cols = check_aligned(got)
    classes = np.array([int(c[2:]) for c in cols])
    models = list(got)
    combos = [(m,) for m in models] + [c for k in range(2, len(models) + 1) for c in itertools.combinations(models, k)]

    rows, thr_rows = [], []
    for combo in combos:
        P_v = np.mean([got[m][0][cols].to_numpy() for m in combo], axis=0)
        P_t = np.mean([got[m][1][cols].to_numpy() for m in combo], axis=0)
        y_v, y_t = got[combo[0]][0].y.to_numpy(), got[combo[0]][1].y.to_numpy()
        name = combo[0] if len(combo) == 1 else "vote(" + "+".join(combo) + ")"
        r = {"구성": name}
        r.update({f"OOF {k}": v for k, v in evaluate(task, y_v, P_v, classes).items()})
        r.update({f"test {k}": v for k, v in evaluate(task, y_t, P_t, classes).items()})
        rows.append(r)
        if task == "binary":
            i1 = list(classes).index(1)
            thr_rows.append(threshold_table(y_v, P_v[:, i1], y_t, P_t[:, i1]).assign(구성=name))

    res = pd.DataFrame(rows)
    display(res.round(4))
    res.to_csv(os.path.join(ENSEMBLE_DIR, f"ensemble_{task}.csv"), index=False)
    ENS[task] = res
    if thr_rows:
        thr_all = pd.concat(thr_rows, ignore_index=True)
        view = thr_all.pivot_table(index="구성", columns="기준", values="test TPR", sort=False)
        print("[이진] 검증에서 정한 임계값을 test에 적용했을 때의 탐지율(TPR)")
        display(view.round(4))
        thr_all.to_csv(os.path.join(ENSEMBLE_DIR, "ensemble_binary_threshold.csv"), index=False)

print("\n저장된 파일:", sorted(os.listdir(ENSEMBLE_DIR)))


===== closed | 확률 파일이 있는 모델: ['lr', 'svm', 'rf']
  [참고] rf: fold 배정이 기준 모델과 다름 (평균 결합에는 문제없음, 스태킹 등에는 사용 불가)


,구성,OOF 정확도,OOF Macro-F1,test 정확도,test Macro-F1
0,lr,0.6630,0.6581,0.6861,0.6815
1,svm,0.8438,0.8442,0.8503,0.8509
2,rf,0.8547,0.8539,0.8600,0.8594
3,vote(lr+svm),0.7840,0.7821,0.8005,0.7989
4,vote(lr+rf),0.7941,0.7917,0.8111,0.8093
5,vote(svm+rf),0.8751,0.8748,0.8876,0.8876
6,vote(lr+svm+rf),0.8412,0.8402,0.8513,0.8503



===== multi | 확률 파일이 있는 모델: ['lr', 'svm', 'rf']
  [참고] rf: fold 배정이 기준 모델과 다름 (평균 결합에는 문제없음, 스태킹 등에는 사용 불가)


,구성,OOF 정확도,OOF Macro-F1,OOF mon 사이트 정답,OOF mon→mon,OOF unmon→unmon,test 정확도,test Macro-F1,test mon 사이트 정답,test mon→mon,test unmon→unmon
0,lr,0.6321,0.5765,0.5526,0.7495,0.7831,0.6338,0.5784,0.5561,0.7489,0.7815
1,svm,0.8205,0.8066,0.7593,0.8434,0.9366,0.8328,0.8238,0.7818,0.8553,0.9295
2,rf,0.8147,0.8043,0.7276,0.7820,0.9802,0.8312,0.8244,0.7503,0.7968,0.9850
3,vote(lr+svm),0.7605,0.7293,0.6799,0.7959,0.9135,0.7760,0.7521,0.7018,0.8018,0.9170
4,vote(lr+rf),0.7553,0.7255,0.6703,0.7731,0.9170,0.7705,0.7476,0.6916,0.7795,0.9205
5,vote(svm+rf),0.8422,0.8374,0.7720,0.8211,0.9756,0.8564,0.8549,0.7918,0.8329,0.9790
6,vote(lr+svm+rf),0.8081,0.7922,0.7289,0.8004,0.9585,0.8221,0.8104,0.7474,0.8095,0.9640



===== binary | 확률 파일이 있는 모델: ['lr', 'svm', 'rf']
  [참고] rf: fold 배정이 기준 모델과 다름 (평균 결합에는 문제없음, 스태킹 등에는 사용 불가)


,구성,OOF 정확도,OOF Macro-F1,OOF ROC-AUC,test 정확도,test Macro-F1,test ROC-AUC
0,lr,0.7810,0.7405,0.8194,0.7745,0.7333,0.8118
1,svm,0.8907,0.8774,0.9487,0.8983,0.8858,0.9524
2,rf,0.8941,0.8783,0.9612,0.9031,0.8888,0.9676
3,vote(lr+svm),0.8822,0.8647,0.9392,0.8903,0.8739,0.9416
4,vote(lr+rf),0.8489,0.8199,0.9356,0.8564,0.8292,0.9402
5,vote(svm+rf),0.9048,0.8918,0.9656,0.9145,0.9030,0.9697
6,vote(lr+svm+rf),0.8949,0.8790,0.9574,0.9012,0.8865,0.9613


[이진] 검증에서 정한 임계값을 test에 적용했을 때의 탐지율(TPR)


기준,기본 (p>0.5),검증 FPR≤0.01,검증 FPR≤0.05,검증 FPR≤0.10,검증 FPR≤0.20
구성,,,,,
lr,0.8908,0.0755,0.2826,0.4555,0.6676
svm,0.9376,0.3792,0.7482,0.8663,0.9439
rf,0.9629,0.5834,0.8145,0.8974,0.9526
vote(lr+svm),0.9547,0.3316,0.6824,0.8326,0.9382
vote(lr+rf),0.9582,0.4611,0.7153,0.8158,0.9024
vote(svm+rf),0.9608,0.6171,0.8363,0.9103,0.9647
vote(lr+svm+rf),0.9624,0.4921,0.7921,0.8942,0.9566



저장된 파일: ['ensemble_binary.csv', 'ensemble_binary_threshold.csv', 'ensemble_closed.csv', 'ensemble_multi.csv', 'rf_converted']


## 결과 해석 시 주의

- 앙상블이 단독 모델보다 **OOF와 test 모두** 나아야 개선으로 본다. test 하나만 좋으면 우연일 수 있다.
- 이진 분류는 수집 흔적의 영향을 받으므로 보조 결과로만 쓴다.
- 가중치는 1:1로 고정했다. 결과를 보고 가중치를 다시 고르면 test로 튜닝하는 셈이 된다.
- 두 모델의 성능 차이가 크면 1:1 평균이 강한 모델 단독보다 낮을 수 있고, 이 경우도 그대로 보고한다.